In [3]:
import sqlite3

import pandas as pd

connection = sqlite3.connect("../data/dublinbikes.db")

## leituras por mes

In [4]:
pd.read_sql_query("""
    SELECT
        strftime('%Y-%m', timestamp_local) AS month,
        COUNT(*) AS total_readings
    FROM readings
    GROUP BY month
    ORDER BY month
""", connection)

,month,total_readings
0,2026-01,470766
1,2026-02,454623
2,2026-03,625668
3,2026-04,617218
4,2026-05,637304
5,2026-06,606511


## as 10 maiores estacoes

In [5]:
pd.read_sql_query("""
    SELECT name, capacity, lat, lon
    FROM stations
    ORDER BY capacity DESC
    LIMIT 10
""", connection)

,name,capacity,lat,lon
0,CONVENTION CENTRE,40,53.347440,-6.238523
1,NEW CENTRAL BANK,40,53.347122,-6.234749
2,THE POINT,40,53.346867,-6.230852
3,GRAND CANAL DOCK,40,53.342636,-6.238695
4,GEORGES LANE,40,53.350230,-6.279696
5,KEVIN STREET,40,53.337757,-6.267699
6,SANDWITH STREET,40,53.345203,-6.247163
7,MOUNT STREET LOWER,40,53.337986,-6.241539
8,YORK STREET WEST,40,53.339333,-6.264699
9,LIME STREET,40,53.346027,-6.243576


## media de bikes por hora do dia

In [6]:
pd.read_sql_query("""
    SELECT
        strftime('%H', timestamp_local) AS hour_of_day,
        ROUND(AVG(num_bikes_available), 2) AS avg_bikes,
        COUNT(*) AS total_readings
    FROM readings
    GROUP BY hour_of_day
    ORDER BY hour_of_day
""", connection)

,hour_of_day,avg_bikes,total_readings
0,00,12.31,124608
1,01,12.32,118832
2,02,12.31,118253
3,03,12.22,125931
4,04,12.32,122648
5,05,12.15,137410
6,06,12.00,143701
7,07,11.51,157749
8,08,11.55,158484
9,09,11.89,147594


## percentual de leituras vazias por estacao

In [7]:
pd.read_sql_query("""
    SELECT
        stations.name,
        stations.capacity,
        COUNT(*) AS total_readings,
        SUM(CASE WHEN readings.num_bikes_available <= 2 THEN 1 ELSE 0 END) AS empty_readings,
        ROUND(
            100.0 * SUM(CASE WHEN readings.num_bikes_available <= 2 THEN 1 ELSE 0 END) / COUNT(*),
            1
        ) AS empty_percent
    FROM readings
    JOIN stations ON stations.station_id = readings.station_id
    GROUP BY stations.station_id, stations.name, stations.capacity
    HAVING COUNT(*) > 10000
    ORDER BY empty_percent DESC
""", connection)

,name,capacity,total_readings,empty_readings,empty_percent
0,HARDWICKE PLACE,25,26138,18666,71.4
1,PARNELL SQUARE NORTH,20,24063,14674,61.0
2,ECCLES STREET EAST,27,26624,16027,60.2
3,DENMARK STREET GREAT,20,26813,15568,58.1
4,FITZWILLIAM SQUARE EAST,40,26592,14324,53.9
...,...,...,...,...,...
110,CUSTOM HOUSE,30,31927,1188,3.7
111,KILMAINHAM GAOL,40,29258,1029,3.5
112,EMMET ROAD,40,28494,980,3.4
113,PRINCES STREET / O'CONNELL STREET,23,34708,1143,3.3
